# Phase M5 — Video B: Train Segmentation Interpolation Model
## Direct Regression U-Net (NOT Diffusion)

**Why not DDPM?** Diffusion models need continuous data + massive datasets. Segmentation masks are discrete labels with tiny tumor regions — DDPM produces noise. Direct regression in one forward pass works much better.

**Architecture:** U-Net takes `seg_start + seg_end + t_interp` → predicts `seg_gt` (cross-entropy loss)

### Kaggle Datasets:
1. **Video A Output** — `mu_glioma_triplets.npz`

---

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 1: Imports & Config
# ═══════════════════════════════════════════════════════════════
import os, time, math, gc, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | GPUs: {torch.cuda.device_count()}')

CFG = {
    'resolution': 128,
    'n_classes': 5,        # labels: 0(bg), 1(NCR), 2(ED), 3(unused), 4(ET)
    'batch_size': 16,
    'lr': 3e-4,
    'epochs': 300,
    'patience': 30,
    'val_split': 0.15,
    'base_ch': 64,
}

OUTPUT_DIR = Path('/kaggle/working/video_train')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Config: {CFG}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 2: Load Data
# ═══════════════════════════════════════════════════════════════
triplets_path = None
for p in Path('/kaggle/input').rglob('mu_glioma_triplets.npz'):
    triplets_path = p; break
if triplets_path is None:
    raise RuntimeError('mu_glioma_triplets.npz not found!')

data = np.load(triplets_path)
seg_start = data['seg_start']  # (N, 128, 128) uint8
seg_end   = data['seg_end']
seg_gt    = data['seg_gt']
t_interp  = data['t_interp']   # (N,) float32
t1c_start = data['t1c_start']  # (N, 128, 128) float32 — brain tissue for overlay
types     = data['types']
N = len(seg_start)
print(f'Loaded {N} samples | Labels: {np.unique(seg_start)}')
print(f'  Real: {(types=="triplet").sum()} | Identity: {(types!="triplet").sum()}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 3: Dataset
# ═══════════════════════════════════════════════════════════════
def seg_to_onehot(seg, C=5):
    oh = np.zeros((C, seg.shape[0], seg.shape[1]), dtype=np.float32)
    for c in range(C):
        oh[c] = (seg == c).astype(np.float32)
    return oh

class SegInterpDataset(Dataset):
    def __init__(self, ss, se, sg, ti, t1c=None, aug=True):
        self.ss, self.se, self.sg, self.ti = ss, se, sg, ti
        self.t1c = t1c
        self.aug = aug

    def __len__(self): return len(self.ss)

    def __getitem__(self, i):
        s = seg_to_onehot(self.ss[i])  # (5, H, W)
        e = seg_to_onehot(self.se[i])
        g = self.sg[i].astype(np.int64)  # (H, W) class labels for CE loss
        t = np.float32(self.ti[i])

        if self.aug:
            if np.random.rand() > 0.5:
                s = s[:, :, ::-1].copy(); e = e[:, :, ::-1].copy()
                g = g[:, ::-1].copy()
            if np.random.rand() > 0.5:
                s = s[:, ::-1].copy(); e = e[:, ::-1].copy()
                g = g[::-1].copy()
            if np.random.rand() > 0.5:
                # Random rotation 90
                k = np.random.choice([1, 2, 3])
                s = np.rot90(s, k, axes=(1,2)).copy()
                e = np.rot90(e, k, axes=(1,2)).copy()
                g = np.rot90(g, k, axes=(0,1)).copy()

        out = {
            'input': torch.from_numpy(np.concatenate([s, e], axis=0)),  # (10, H, W)
            't_interp': torch.tensor([t]),
            'gt': torch.from_numpy(g),  # (H, W) long
        }
        if self.t1c is not None:
            out['t1c'] = torch.from_numpy(self.t1c[i])
        return out

np.random.seed(42)
idx = np.random.permutation(N)
nv = int(N * CFG['val_split'])
train_ds = SegInterpDataset(seg_start[idx[nv:]], seg_end[idx[nv:]],
                            seg_gt[idx[nv:]], t_interp[idx[nv:]],
                            t1c=t1c_start[idx[nv:]], aug=True)
val_ds   = SegInterpDataset(seg_start[idx[:nv]], seg_end[idx[:nv]],
                            seg_gt[idx[:nv]], t_interp[idx[:nv]],
                            t1c=t1c_start[idx[:nv]], aug=False)
train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                      num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Batches: {len(train_dl)}/ep')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 4: U-Net Architecture (Direct Regression)
# ═══════════════════════════════════════════════════════════════

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim=128):
        super().__init__()
        # Use num_groups that divides in_ch
        g1 = min(8, in_ch) if in_ch % 8 == 0 else (4 if in_ch % 4 == 0 else 1)
        g2 = min(8, out_ch) if out_ch % 8 == 0 else (4 if out_ch % 4 == 0 else 1)
        self.conv1 = nn.Sequential(nn.GroupNorm(g1, in_ch), nn.SiLU(),
                                   nn.Conv2d(in_ch, out_ch, 3, 1, 1))
        self.conv2 = nn.Sequential(nn.GroupNorm(g2, out_ch), nn.SiLU(),
                                   nn.Conv2d(out_ch, out_ch, 3, 1, 1))
        self.t_proj = nn.Sequential(nn.SiLU(), nn.Linear(t_dim, out_ch))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.skip(x)

class SegInterpUNet(nn.Module):
    """
    Direct regression U-Net.
    Input:  seg_start(5ch) + seg_end(5ch) = 10 channels
    Condition: t_interp (scalar, via FiLM embedding)
    Output: logits (5ch) → argmax → predicted segmentation
    """
    def __init__(self, in_ch=10, out_ch=5, base=64, t_dim=128):
        super().__init__()
        self.t_embed = nn.Sequential(
            nn.Linear(1, t_dim), nn.SiLU(),
            nn.Linear(t_dim, t_dim), nn.SiLU()
        )
        self.input_proj = nn.Sequential(nn.Conv2d(in_ch, base, 3, 1, 1), nn.SiLU())

        self.enc1 = ResBlock(base, base, t_dim)
        self.enc2 = ResBlock(base, base*2, t_dim)
        self.enc3 = ResBlock(base*2, base*4, t_dim)
        self.enc4 = ResBlock(base*4, base*8, t_dim)
        self.down = nn.MaxPool2d(2)

        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec3 = ResBlock(base*8 + base*4, base*4, t_dim)
        self.dec2 = ResBlock(base*4 + base*2, base*2, t_dim)
        self.dec1 = ResBlock(base*2 + base, base, t_dim)
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x, t_interp):
        t = self.t_embed(t_interp)
        x = self.input_proj(x)
        e1 = self.enc1(x, t)
        e2 = self.enc2(self.down(e1), t)
        e3 = self.enc3(self.down(e2), t)
        e4 = self.enc4(self.down(e3), t)
        d3 = self.dec3(torch.cat([self.up(e4), e3], 1), t)
        d2 = self.dec2(torch.cat([self.up(d3), e2], 1), t)
        d1 = self.dec1(torch.cat([self.up(d2), e1], 1), t)
        return self.out(d1)  # (B, 5, H, W) logits

model = SegInterpUNet(in_ch=10, out_ch=CFG['n_classes'], base=CFG['base_ch']).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'SegInterpUNet: {n_params:.1f}M params')

# Shape test
with torch.no_grad():
    x = torch.randn(2, 10, 128, 128).to(device)
    t = torch.rand(2, 1).to(device)
    o = model(x, t)
    print(f'Shape: ({2},10,128,128) → {o.shape} ✅')
    del x, t, o; torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 5: Loss — Dice + Cross-Entropy
# ═══════════════════════════════════════════════════════════════

def dice_loss(logits, target, smooth=1.0):
    """Soft Dice loss per class, averaged."""
    probs = torch.softmax(logits, dim=1)  # (B, C, H, W)
    target_oh = F.one_hot(target, num_classes=CFG['n_classes']).permute(0, 3, 1, 2).float()
    dims = (2, 3)
    inter = (probs * target_oh).sum(dims)
    union = probs.sum(dims) + target_oh.sum(dims)
    dice = (2 * inter + smooth) / (union + smooth)
    return 1 - dice[:, 1:].mean()  # skip background

def combined_loss(logits, target):
    ce = F.cross_entropy(logits, target, weight=torch.tensor([0.1, 1.0, 1.0, 0.1, 1.5], device=device))
    dl = dice_loss(logits, target)
    return ce + dl

def compute_dice(pred_labels, gt_labels):
    """Compute Dice for WT, TC, ET."""
    results = {}
    wt_p, wt_g = pred_labels > 0, gt_labels > 0
    tc_p, tc_g = (pred_labels == 1) | (pred_labels == 4), (gt_labels == 1) | (gt_labels == 4)
    et_p, et_g = pred_labels == 4, gt_labels == 4
    for name, p, g in [('WT', wt_p, wt_g), ('TC', tc_p, tc_g), ('ET', et_p, et_g)]:
        inter = (p & g).sum().float()
        union = p.sum().float() + g.sum().float()
        results[name] = (2 * inter / (union + 1e-8)).item()
    return results

print('Loss functions ready (Dice + CE)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 6: Training Loop
# ═══════════════════════════════════════════════════════════════
from torch.cuda.amp import GradScaler, autocast

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)
scaler = GradScaler()

best_val_loss = float('inf')
best_val_dice = 0.0
patience_ct = 0
train_losses, val_losses, val_dices = [], [], []
CKPT = OUTPUT_DIR / 'seg_interp_best.pth'

# Resume
start_epoch = 0
for p in Path('/kaggle/input').rglob('seg_interp_best.pth'):
    ckpt = torch.load(p, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    if 'epoch' in ckpt: start_epoch = ckpt['epoch'] + 1
    if 'best_val_dice' in ckpt: best_val_dice = ckpt['best_val_dice']
    print(f'Resumed from ep {start_epoch}, best_dice={best_val_dice:.4f}')
    break

print(f'\nTraining: {start_epoch}→{CFG["epochs"]}, patience={CFG["patience"]}')
t0 = time.time()

for epoch in range(start_epoch, CFG['epochs']):
    model.train()
    ep_loss = []
    for batch in train_dl:
        x  = batch['input'].to(device)
        ti = batch['t_interp'].to(device)
        gt = batch['gt'].to(device)

        with autocast():
            logits = model(x, ti)
            loss = combined_loss(logits, gt)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        ep_loss.append(loss.item())

    scheduler.step()
    train_loss = np.mean(ep_loss)
    train_losses.append(train_loss)

    # Validation
    model.eval()
    vl, vd = [], {'WT': [], 'TC': [], 'ET': []}
    with torch.no_grad():
        for batch in val_dl:
            x  = batch['input'].to(device)
            ti = batch['t_interp'].to(device)
            gt = batch['gt'].to(device)
            with autocast():
                logits = model(x, ti)
                vl.append(combined_loss(logits, gt).item())
            pred = logits.argmax(dim=1)
            for b in range(pred.shape[0]):
                d = compute_dice(pred[b], gt[b])
                for k in d: vd[k].append(d[k])

    val_loss = np.mean(vl)
    val_losses.append(val_loss)
    mean_dice = np.mean([np.mean(vd[k]) for k in vd])
    val_dices.append(mean_dice)

    improved = mean_dice > best_val_dice
    if improved:
        best_val_dice = mean_dice
        best_val_loss = val_loss
        patience_ct = 0
        torch.save({'model': model.state_dict(), 'epoch': epoch,
                     'best_val_dice': best_val_dice, 'cfg': CFG}, str(CKPT))
    else:
        patience_ct += 1

    if (epoch+1) % 10 == 0 or improved:
        wt_d = np.mean(vd['WT']); tc_d = np.mean(vd['TC']); et_d = np.mean(vd['ET'])
        print(f'  Ep {epoch+1:3d}/{CFG["epochs"]} | loss={train_loss:.4f}/{val_loss:.4f} | '
              f'Dice WT={wt_d:.3f} TC={tc_d:.3f} ET={et_d:.3f} avg={mean_dice:.3f} | '
              f'best={best_val_dice:.3f} pat={patience_ct} | {time.time()-t0:.0f}s')

    if patience_ct >= CFG['patience']:
        print(f'  Early stop at epoch {epoch+1}')
        break

print(f'\n✅ Done. Best avg Dice: {best_val_dice:.4f} | Checkpoint: {CKPT}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 7: Training Curves
# ═══════════════════════════════════════════════════════════════
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, label='Train', color='#1565C0')
ax1.plot(val_losses, label='Val', color='#E53935')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss (CE+Dice)')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(val_dices, label='Avg Dice', color='#2E7D32', lw=2)
ax2.axhline(best_val_dice, ls='--', color='gray', label=f'Best: {best_val_dice:.3f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice Score')
ax2.set_title('Validation Dice'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()
print('✅ Curves saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 8: Visual Sampling Test (Overlaid on Brain Tissue)
# ═══════════════════════════════════════════════════════════════
ckpt = torch.load(str(CKPT), map_location=device, weights_only=False)
model.load_state_dict(ckpt['model']); model.eval()

def label_to_rgb(seg):
    rgb = np.zeros((*seg.shape, 3), dtype=np.float32)
    rgb[seg == 1] = [0.2, 0.4, 1.0]  # NCR blue
    rgb[seg == 2] = [0.2, 0.8, 0.3]  # ED green
    rgb[seg == 4] = [1.0, 0.2, 0.2]  # ET red
    return rgb

def overlay_seg_on_mri(seg, t1c, alpha=0.6):
    """Overlay colored segmentation on grayscale T1c MRI."""
    t1c_rgb = np.stack([t1c]*3, axis=-1)  # grayscale → RGB
    seg_rgb = label_to_rgb(seg)
    mask = seg_rgb.sum(axis=-1, keepdims=True) > 0  # where tumor exists
    overlay = np.where(mask, (1-alpha)*t1c_rgb + alpha*seg_rgb, t1c_rgb)
    return np.clip(overlay, 0, 1)

# Test on 4 val samples
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('Segmentation Interpolation on Brain MRI — Val Samples', fontsize=14, fontweight='bold')
cols = ['Start (t=0)', 'Ground Truth', 'Predicted', 'End (t=1)']
for j, c in enumerate(cols):
    axes[0, j].set_title(c, fontsize=12, fontweight='bold')

for row in range(4):
    sample = val_ds[row * 5]
    x  = sample['input'].unsqueeze(0).to(device)
    ti = sample['t_interp'].unsqueeze(0).to(device)
    gt = sample['gt'].numpy()
    t_val = sample['t_interp'].item()
    t1c = sample['t1c'].numpy() if 't1c' in sample else np.zeros((128,128))

    with torch.no_grad():
        pred = model(x, ti).argmax(dim=1)[0].cpu().numpy()

    start_lbl = sample['input'][:5].argmax(dim=0).numpy()
    end_lbl   = sample['input'][5:].argmax(dim=0).numpy()

    d = compute_dice(torch.tensor(pred), torch.tensor(gt))

    for j, (seg, title) in enumerate(zip(
        [start_lbl, gt, pred, end_lbl],
        ['t=0', f't={t_val:.2f} (GT)', f't={t_val:.2f} (Pred)', 't=1'])):
        axes[row, j].imshow(overlay_seg_on_mri(seg, t1c), origin='lower')
        axes[row, j].axis('off')
    axes[row, 2].set_xlabel(f'WT={d["WT"]:.2f}  TC={d["TC"]:.2f}  ET={d["ET"]:.2f}', fontsize=9)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'sampling_test.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sampling test saved (overlaid on MRI)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 9: Interpolation Sequence on Brain MRI
# ═══════════════════════════════════════════════════════════════
sample = val_ds[0]
start_oh = sample['input'][:5].unsqueeze(0).to(device)
end_oh   = sample['input'][5:].unsqueeze(0).to(device)
t1c = sample['t1c'].numpy() if 't1c' in sample else np.zeros((128,128))

fig, axes = plt.subplots(1, 11, figsize=(22, 2.5))
fig.suptitle('Tumor Evolution: t=0.0 → t=1.0 (overlaid on MRI)', fontsize=13, fontweight='bold')

for i, t_val in enumerate(np.linspace(0, 1, 11)):
    inp = torch.cat([start_oh, end_oh], dim=1)
    ti  = torch.tensor([[t_val]], device=device)
    with torch.no_grad():
        pred = model(inp, ti).argmax(dim=1)[0].cpu().numpy()
    axes[i].imshow(overlay_seg_on_mri(pred, t1c), origin='lower')
    axes[i].set_title(f't={t_val:.1f}', fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'interpolation_sequence.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Interpolation sequence saved (on MRI)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 10: Summary
# ═══════════════════════════════════════════════════════════════
print('\n' + '═'*60)
print('  SEGMENTATION INTERPOLATION — COMPLETE')
print('═'*60)
print(f'  Model:      SegInterpUNet ({n_params:.1f}M params)')
print(f'  Train/Val:  {len(train_ds)}/{len(val_ds)}')
print(f'  Best Dice:  {best_val_dice:.4f}')
print(f'  Epochs:     {len(train_losses)}')
print(f'\n  Output files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'    {f.name:40s} {f.stat().st_size/1e6:7.1f} MB')
print(f'\n  → Next: Video_C_Validate.ipynb')
print('═'*60)